# Módulo F3 · Estadística para la toma de decisiones — Soluciones

Este notebook resuelve los cuatro ejercicios del final de `notebook.ipynb`. Es **autocontenido**: primero reproduce la preparación del notebook original (misma semilla y mismos pasos aleatorios), de modo que la población, la muestra de 200 pedidos y sus resultados coinciden con los que viste allí.

> Consejo: intenta resolver cada ejercicio por tu cuenta antes de leer la solución. Lo importante no es el número exacto, sino el razonamiento y la interpretación.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(7)

## Preparación: reproducimos los pasos aleatorios del notebook

El generador `rng` es una secuencia: cada llamada consume números aleatorios. Para obtener **exactamente** la misma población y la misma muestra de 200 pedidos que en el notebook, repetimos las mismas llamadas en el mismo orden (sin dibujar los gráficos).

In [ ]:
# Sección 1 del notebook: muestras de las distribuciones (solo para avanzar el generador)
_ = rng.binomial(20, 0.1, 10_000); _ = rng.poisson(4, 10_000)
_ = rng.exponential(5, 10_000); _ = rng.lognormal(3.2, 0.5, 10_000)

# Sección 2: la "población" de 1 millón de pedidos
poblacion = rng.lognormal(mean=3.2, sigma=0.5, size=1_000_000)
mu_real = poblacion.mean()

# Sección 3: ley de los grandes números
_ = rng.choice(poblacion, 5_000)

# Sección 4: TCL
for n in [2, 10, 100]:
    _ = rng.choice(poblacion, (5_000, n))

# Sección 5: la muestra de 200 pedidos y su IC
muestra = rng.choice(poblacion, 200)
media, ee = muestra.mean(), stats.sem(muestra)
ic = stats.t.interval(0.95, df=len(muestra) - 1, loc=media, scale=ee)

print(f"Media poblacional (desconocida en la práctica): {mu_real:.2f} €")
print(f"Muestra de n = {len(muestra)} | media = {media:.2f} € | s = {muestra.std(ddof=1):.2f} € | EE = {ee:.2f} €")
print(f"IC 95 % (t de Student): [{ic[0]:.2f}, {ic[1]:.2f}]")

A partir de aquí usamos un generador nuevo para los ejercicios, así los resultados son reproducibles sin depender del orden de las celdas anteriores.

In [ ]:
rng_ej = np.random.default_rng(2026)

def bootstrap(datos, estadistico, n_boot=5_000, rng=rng_ej):
    """Distribución bootstrap de un estadístico (remuestreo con reemplazo)."""
    idx = rng.integers(0, len(datos), (n_boot, len(datos)))
    return np.array([estadistico(datos[i]) for i in idx])

---
## Ejercicio 1 · Cobertura del IC con muestras de `n = 15`

*Repite el experimento de cobertura con muestras de `n = 15`. ¿Sigue siendo ~95 %? ¿Por qué podría fallar con datos tan asimétricos?*

Con 100 intervalos, como en el notebook, la cobertura observada oscila mucho por puro azar (es un recuento binomial: 95 ± ~4). Para medirla con precisión repetimos el experimento **10 000 veces** para varios tamaños de muestra. Vectorizamos el cálculo para que sea rápido.

In [ ]:
def cobertura(n, n_rep=10_000, nivel=0.95, rng=rng_ej):
    """Simula n_rep muestras de tamaño n y devuelve la cobertura del IC t
    y en qué lado de la media real quedan los intervalos fallidos."""
    muestras = rng.choice(poblacion, (n_rep, n))
    medias = muestras.mean(axis=1)
    ees = muestras.std(axis=1, ddof=1) / np.sqrt(n)
    t = stats.t.ppf(0.5 + nivel / 2, df=n - 1)
    lo, hi = medias - t * ees, medias + t * ees
    return {
        "n": n,
        "cobertura": np.mean((lo <= mu_real) & (mu_real <= hi)),
        "fallo: IC por debajo de μ": np.mean(hi < mu_real),
        "fallo: IC por encima de μ": np.mean(lo > mu_real),
        "semianchura media (€)": np.mean(t * ees),
    }

res_cob = pd.DataFrame([cobertura(n) for n in [5, 15, 50, 200]]).set_index("n")
res_cob.round(3)

Veamos los 100 primeros intervalos con `n = 15`, igual que el gráfico del notebook:

In [ ]:
m15 = rng_ej.choice(poblacion, (100, 15))
med15 = m15.mean(axis=1)
ee15 = m15.std(axis=1, ddof=1) / np.sqrt(15)
t15 = stats.t.ppf(0.975, df=14)
lo15, hi15 = med15 - t15 * ee15, med15 + t15 * ee15
contiene = (lo15 <= mu_real) & (mu_real <= hi15)

plt.figure(figsize=(12, 3))
for i in range(100):
    plt.plot([i, i], [lo15[i], hi15[i]], color="tab:blue" if contiene[i] else "red")
plt.axhline(mu_real, color="black", ls="--")
plt.title(f"n = 15: {contiene.sum()} de 100 intervalos contienen la media real")
plt.show()

¿Por qué falla? La fórmula $\bar x \pm t\cdot s/\sqrt n$ supone que la media muestral es aproximadamente normal. Con `n = 15` y datos log-normales, la distribución de $\bar x$ aún tiene **cola derecha**, y además $\bar x$ y $s$ están **correlacionados**: las muestras que no han «pillado» ningún pedido grande tienen media baja *y* desviación pequeña, así que su intervalo es estrecho y se queda corto por debajo de la media real.

In [ ]:
m = rng_ej.choice(poblacion, (10_000, 15))
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].hist(m.mean(axis=1), bins=60, density=True, alpha=0.7)
x = np.linspace(m.mean(axis=1).min(), m.mean(axis=1).max(), 200)
axes[0].plot(x, stats.norm.pdf(x, mu_real, poblacion.std() / np.sqrt(15)), "r", label="normal del TCL")
axes[0].set_title("Distribución de la media con n = 15 (asimetría residual)"); axes[0].legend()
axes[1].scatter(m.mean(axis=1)[:2000], m.std(axis=1, ddof=1)[:2000], s=4, alpha=0.4)
axes[1].axvline(mu_real, color="red", ls="--")
axes[1].set_xlabel("media muestral"); axes[1].set_ylabel("desviación típica muestral s")
r = np.corrcoef(m.mean(axis=1), m.std(axis=1, ddof=1))[0, 1]
axes[1].set_title(f"Media y s van juntas (r = {r:.2f})")
plt.tight_layout(); plt.show()
print(f"Asimetría de la media muestral con n = 15: {stats.skew(m.mean(axis=1)):.2f} (0 en una normal)")

**Interpretación.** Los números exactos dependen de la simulación, pero el patrón es claro en la tabla:

- Con `n = 15` la cobertura real queda **por debajo del 95 % nominal** (alrededor del 93 %) y con `n = 5` aún peor (≈ 92 %); con `n = 200` vuelve a estar muy cerca del 95 %.
- Los fallos **no son simétricos**: casi todos son intervalos que se quedan *por debajo* de la media real. Con datos asimétricos a la derecha, las muestras pequeñas suelen no incluir los pedidos grandes, subestiman la media y además estiman una $s$ pequeña, lo que produce intervalos estrechos y desplazados.
- El TCL es un resultado **asintótico**: cuanto más asimétrica es la distribución, más grande debe ser `n` para que la aproximación normal funcione bien. No hay un «n = 30 mágico».

**Qué harías en la práctica con muestras pequeñas y asimétricas:** usar más datos si es posible; usar un intervalo bootstrap (mejor aún, BCa) o trabajar en escala logarítmica; y, sobre todo, comunicar que la incertidumbre real es algo mayor de lo que dice la fórmula.

---
## Ejercicio 2 · IC bootstrap para el percentil 90 del importe

*Calcula por bootstrap un IC 95 % para el **percentil 90** del importe. ¿Por qué sería útil para planificar repartidores?*

In [ ]:
p90 = lambda x: np.percentile(x, 90)

boot_p90 = bootstrap(muestra, p90)
lo90, hi90 = np.percentile(boot_p90, [2.5, 97.5])
print(f"P90 muestral: {p90(muestra):.2f} € | IC 95 % bootstrap (percentil): [{lo90:.2f}, {hi90:.2f}]")
print(f"Error estándar bootstrap del P90: {boot_p90.std(ddof=1):.2f} €")
print(f"P90 poblacional (desconocido en la práctica): {p90(poblacion):.2f} €")

# Comparación: la misma idea con SciPy (método BCa, corrige sesgo y asimetría)
res_bca = stats.bootstrap((muestra,), p90, n_resamples=5_000, method="BCa", random_state=2026)
print(f"IC 95 % BCa (scipy.stats.bootstrap): [{res_bca.confidence_interval.low:.2f}, {res_bca.confidence_interval.high:.2f}]")

In [ ]:
plt.figure(figsize=(8, 3.5))
plt.hist(boot_p90, bins=50, alpha=0.75)
plt.axvline(lo90, color="k", ls="--"); plt.axvline(hi90, color="k", ls="--", label="IC 95 % bootstrap")
plt.axvline(p90(poblacion), color="red", label="P90 poblacional")
plt.title("Distribución bootstrap del percentil 90"); plt.xlabel("€"); plt.legend(); plt.show()
print(f"Valores distintos del P90 entre los {len(boot_p90)} remuestreos: {len(np.unique(np.round(boot_p90, 4)))}")

**Interpretación.**

- El intervalo bootstrap del P90 es **mucho más ancho en euros** que el de la media (el error estándar bootstrap del P90, ≈ 3,9 €, es más del triple que el de la media, ≈ 1,14 €). Es lógico: el P90 depende de las ~20 observaciones más altas de la muestra, que son pocas y muy variables. Estimar colas exige más datos que estimar el centro.
- La distribución bootstrap es **irregular, «a saltos»**: el percentil de una remuestra solo puede tomar valores cercanos a los de la muestra original. Es una señal de que con `n = 200` el bootstrap de percentiles extremos es tosco; con percentiles aún más extremos (P99) no sería fiable.
- En esta ejecución el intervalo sí contiene el P90 poblacional, pero recuerda que eso solo lo sabemos porque estamos simulando.

**¿Por qué es útil para operaciones?** La media dice cuánto factura un pedido «típico», pero la planificación se rompe con los pedidos grandes: más bultos, más peso, a veces dos viajes o un vehículo con más capacidad, más tiempo de entrega y más riesgo si hay que cobrar en efectivo. El P90 responde a «¿a partir de qué importe está el 10 % de pedidos más grandes?», y su **límite superior del IC** es un valor prudente para dimensionar: cuántas mochilas grandes o furgonetas tener por turno, qué umbral usar para asignar automáticamente un vehículo mayor, o qué límite de cobro en efectivo fijar. Planificar con la media dejaría infradimensionado justo el tramo que genera retrasos.

---
## Ejercicio 3 · Tamaño de muestra para una semianchura de ±0,50 €

*¿Qué tamaño de muestra necesitas para que el IC de la media tenga una semianchura de ±0,50 €?*

Despejamos $n$ de $1{,}96\, s/\sqrt n = E$:
$$n = \left(\frac{1{,}96\, s}{E}\right)^2$$
usando como $s$ la desviación típica de nuestra muestra piloto de 200 pedidos.

In [ ]:
s = muestra.std(ddof=1)
E = 0.50
n_necesario = int(np.ceil((1.96 * s / E) ** 2))
print(f"s (muestra piloto) = {s:.2f} €  ->  n necesario ≈ {n_necesario:,} pedidos".replace(",", "."))

# Sensibilidad: s también es una estimación con incertidumbre. Un IC bootstrap para s
boot_s = bootstrap(muestra, lambda x: x.std(ddof=1))
s_lo, s_hi = np.percentile(boot_s, [2.5, 97.5])
for etiqueta, s_ in [("s bajo (2,5 %)", s_lo), ("s estimado", s), ("s alto (97,5 %)", s_hi)]:
    n_s = f"{int(np.ceil((1.96 * s_ / E) ** 2)):,}".replace(",", ".")
    print(f"{etiqueta:>16}: s = {s_:5.2f} €  ->  n = {n_s:>6}")

Comprobamos por simulación (sobre la población, que aquí sí conocemos) qué semianchura se obtiene con ese $n$, y vemos cómo decrece con $n$ (regla de $1/\sqrt n$):

In [ ]:
tamanos = [200, 800, 1_000, 2_000, n_necesario]
filas = []
for n in tamanos:
    sims = rng_ej.choice(poblacion, (300, n))
    semi = stats.t.ppf(0.975, n - 1) * sims.std(axis=1, ddof=1) / np.sqrt(n)
    filas.append({"n": n, "semianchura media (€)": semi.mean(), "p95 de la semianchura (€)": np.percentile(semi, 95)})
pd.DataFrame(filas).set_index("n").round(3)

**Interpretación.**

- Con la muestra piloto ($s \approx 16{,}1$ €) necesitamos del orden de **4 000 pedidos** para un IC 95 % de ±0,50 €. Con nuestros 200 pedidos la semianchura es ±2,25 €.
- La relación es cuadrática: **reducir el margen a la mitad exige cuadruplicar la muestra** (en la simulación, de 200 a 800 pedidos la semianchura media pasa de ≈ 2,05 € a ≈ 1,03 €).
- En la simulación, con 3 996 pedidos obtenemos ≈ ±0,46 €, algo mejor de lo previsto: la desviación típica real de la población (≈ 14,8 €) es menor que la $s$ de nuestra muestra piloto (16,1 €). Es justo la lección: **el cálculo depende de $s$, que es solo una estimación**. Si la $s$ real estuviera en la parte alta de su IC bootstrap (≈ 18,6 €), harían falta unos 5 300 pedidos; si estuviera en la parte baja (≈ 13,6 €), bastarían unos 2 800. Una regla razonable es redondear al alza (p. ej., **4 500–5 000 pedidos**) o recalcular a mitad de recogida.
- Pregunta de negocio: ¿de verdad necesitamos ±0,50 €? Si la decisión (p. ej., fijar el pedido mínimo) no cambia entre 27,5 € y 28,5 €, una muestra de 800–1 000 pedidos puede bastar. El tamaño de muestra se decide por **el coste de equivocarse**, no por perfeccionismo.

---
## Ejercicio 4 · Otro ejemplo de paradoja de Simpson

*Inventa otro ejemplo de paradoja de Simpson en tu sector y escríbelo en una tabla como la anterior.*

**Respuesta modelo (logística/reparto).** Comparamos la tasa de entregas a tiempo de dos flotas: la **propia** y una **subcontratada**. La flota propia trabaja sobre todo en la periferia (rutas largas, más retrasos); la subcontratada, sobre todo en el centro.

In [ ]:
entregas = pd.DataFrame({
    "flota": ["propia", "propia", "subcontratada", "subcontratada"],
    "zona":  ["centro", "periferia", "centro", "periferia"],
    "entregas": [1_000, 4_000, 4_000, 1_000],
    "a_tiempo": [950, 3_200, 3_600, 750],
})
entregas["tasa"] = entregas["a_tiempo"] / entregas["entregas"]
print(entregas, "\n")
glob_e = entregas.groupby("flota")[["entregas", "a_tiempo"]].sum()
print("Tasa global de entregas a tiempo:\n", (glob_e["a_tiempo"] / glob_e["entregas"]).round(3), sep="")

In [ ]:
piv = entregas.pivot(index="zona", columns="flota", values="tasa")
piv.loc["GLOBAL"] = glob_e["a_tiempo"] / glob_e["entregas"]
ax = piv.plot.bar(figsize=(7, 3.5), rot=0)
ax.set_ylim(0.7, 1); ax.set_ylabel("tasa de entregas a tiempo")
ax.set_title("La flota propia gana en cada zona... y pierde en el global"); plt.show()

**Interpretación.**

| | Centro | Periferia | Global |
|---|---|---|---|
| Propia | **95,0 %** (1 000) | **80,0 %** (4 000) | 83,0 % |
| Subcontratada | 90,0 % (4 000) | 75,0 % (1 000) | **87,0 %** |

- En el global, la subcontratada parece mejor (87 % frente a 83 %), y alguien podría proponer externalizar más rutas.
- Pero **en cada zona la flota propia es 5 puntos mejor**. La inversión se debe a que la flota propia hace el 80 % de su trabajo en la periferia, donde cualquier flota llega tarde más a menudo. La **zona** es una variable de confusión: influye a la vez en qué flota se asigna y en la puntualidad.
- ¿Qué cifra usar? Para la pregunta «¿qué flota es más puntual *en igualdad de condiciones*?», la comparación por zona (o una tasa ajustada que use la misma mezcla de zonas para ambas flotas). La cifra global solo describe la mezcla actual de rutas, no la calidad de la flota.
- Ojo: estratificar por zona no garantiza que no haya **otras** confusoras (tipo de vehículo, franja horaria, tamaño del pedido). La única forma de descartarlas todas es asignar las rutas **al azar** durante un periodo de prueba (diseño experimental, módulo 04).

---
## Extra · Borrador modelo del caso práctico (fragmento)

Un ejemplo de cómo trasladar los resultados a lenguaje llano para el director comercial:

> **Importe medio del pedido: 28,1 € (entre 25,8 € y 30,3 € con un 95 % de confianza).**
> Lo hemos estimado con una muestra aleatoria de 200 pedidos. El margen de ±2,2 € refleja que no hemos visto todos los pedidos: si repitiéramos el análisis con otras muestras de 200, nuestro método acertaría (el intervalo incluiría el valor real) en 95 de cada 100 ocasiones. El dato es suficiente para decisiones gruesas (p. ej., distinguir un ticket de 28 € de uno de 35 €), pero no para detectar cambios de menos de ~2 €.
> **Recomendación:** si queremos medir el efecto de un cambio de interfaz con precisión de ±0,50 €, necesitamos unos 4 000–5 000 pedidos por grupo y asignar las versiones al azar. Además, la mitad de los pedidos está por debajo de 24,5 € (la mediana): unos pocos pedidos grandes elevan la media, por lo que conviene reportar ambas cifras.